# Claim Extractor Test Notebook

This notebook tests the `extract_claims` function from `src/claim_extraction/extractor.py` with two local model runs and one OpenAI remote model run.

Models are loaded from `.env`, for example:
- `CLAIM_MODEL_1=Qwen/Qwen3.5-0.8B`
- `CLAIM_MODEL_2=Qwen/Qwen2.5-0.5B-Instruct`

## Goals
- Compare two local models and one remote (OpenAI) model
- Use direct prompt-based extraction only
- Quickly inspect claim count and claim content

## How extract_claims function should be called
claims = extract_claims(
    text=story,
    model_name=model_name,
    backend=backend,
    temperature=temperature,
    verbose=verbose
)

In [1]:
import textwrap

from dotenv import dotenv_values

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims

env = dotenv_values(DOTENV_PATH)

In [2]:
# Centralized config from .env (with defaults)
LOCAL_MODEL_1 = env.get('CLAIM_MODEL_1', 'Qwen/Qwen3.5-4B')
LOCAL_MODEL_2 = env.get('CLAIM_MODEL_2', 'Qwen/Qwen2.5-3B-Instruct')
REMOTE_MODEL = env.get('CLAIM_MODEL_REMOTE', 'gpt-4o-mini')
openai_api_key = env.get('OPENAI_API_KEY', '')

# Standard generation parameters used by all runs
TEMPERATURE = 0.1
VERBOSE = True

print('LOCAL_MODEL_1:', LOCAL_MODEL_1)
print('LOCAL_MODEL_2:', LOCAL_MODEL_2)
print('REMOTE_MODEL:', REMOTE_MODEL)
print('OPENAI_API_KEY:', '<set>' if openai_api_key else None)
print('TEMPERATURE:', TEMPERATURE)
print('VERBOSE:', VERBOSE)

LOCAL_MODEL_1: Qwen/Qwen3.5-4B
LOCAL_MODEL_2: Qwen/Qwen2.5-3B-Instruct
REMOTE_MODEL: gpt-4o-mini
OPENAI_API_KEY: <set>
TEMPERATURE: 0.1
VERBOSE: True


# Get longest story from ContraDoc

In [3]:
import json
from pathlib import Path

# Find the longest story in the test set for a more challenging extraction task
data_path = Path("datasets/ContraDoc/ContraDoc.json")
with data_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

def to_record_list(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        # Common split containers in contradiction datasets
        split_keys = ("test", "train", "validation", "data", "examples", "pos", "neg")
        records = []
        for key in split_keys:
            if key in obj:
                records.extend(to_record_list(obj[key]))
        if records:
            return records
        return list(obj.values())
    return []

records = to_record_list(data)

def extract_text(record):
    if isinstance(record, str):
        return record
    if isinstance(record, dict):
        for key in ("story", "text", "document", "doc", "passage", "content"):
            value = record.get(key)
            if isinstance(value, str):
                return value
        str_values = [v for v in record.values() if isinstance(v, str)]
        return max(str_values, key=len, default="")
    return ""

story = max((extract_text(r) for r in records), key=len, default="")

# Alternative, shorter story below

# story = textwrap.dedent('''
# In the valley of Meridia, a mouse named Tim wore a red brass-button coat and carried a map that his grandmother had drawn in 1984.
# Tim claimed the map showed three bridges over the river, but the mayor had announced last winter that only two bridges remained after flooding.
# At dawn, engineer Nora inspected the eastern bridge and told the council that the structure was safe for carts lighter than 500 kilograms.
# During the same meeting, historian Elias argued that the eastern bridge had already collapsed in 1999.
# Tim then said that if the eastern bridge was really safe, then every farmer from Oak District would deliver grain by sunset.
# By noon, two farmers from Oak District delivered grain, while four others reported blocked roads.
# In the afternoon, meteorologist Lina predicted heavy rain by evening, and she added that if rainfall exceeded 30 millimeters, then the western bridge would close automatically.
# The rain gauge later measured 34 millimeters.
# Nevertheless, the western bridge remained open until midnight according to the city logbook.
# Separately, archivist Omar wrote that there exists a lighthouse keeper named Rae who had never visited Meridia, even though a travel diary signed by Rae described a market visit in Meridia in 2022.
# At the festival, the announcer said the town had exactly 120 lanterns, while the inventory sheet listed 118 lanterns and two repaired frames.
# Finally, council minutes stated that if the budget was approved on Tuesday, then the school roof would be repaired before October, but the budget vote was postponed to Thursday.
# ''').strip()

print(f"Loaded records: {len(records)}")
print(f"Input length: {len(story)} chars")

Loaded records: 891
Input length: 9330 chars


In [4]:
# Re-usable function to run a claim extraction case with error handling and consistent output 
# formatting

def run_case(label, model_name, backend, temperature, verbose):
    print('\n' + '=' * 100)
    print(label)
    print('=' * 100)
    try:
        claims = extract_claims(
            text=story,
            model_name=model_name,
            backend=backend,
            temperature=temperature,
            verbose=verbose,
        )
        print(f'Claim count: {len(claims)}')
        for i, c in enumerate(claims, 1):
            print(f'{i:02d}. {c}')
        return claims
    except Exception as exc:
        print(f'Run failed: {exc}')
        return []

In [5]:
# # Direct extraction, local model 1
# direct_local_model_1_claims = run_case(
#     label=f'Direct | local | {LOCAL_MODEL_1}',
#     model_name=LOCAL_MODEL_1,
#     backend='local',
#     temperature=TEMPERATURE,
#     verbose=VERBOSE,
# )

In [6]:
# # Direct extraction, local model 2
# direct_local_model_2_claims = run_case(
#     label=f'Direct | local | {LOCAL_MODEL_2}',
#     model_name=LOCAL_MODEL_2,
#     backend='local',
#     temperature=TEMPERATURE,
#     verbose=VERBOSE,
# )

## OpenAI Remote Test

This test uses the remote OpenAI-compatible backend (`backend='remote'`).

Expected in `.env`:
- `OPENAI_API_KEY`
- `CLAIM_MODEL_REMOTE` (optional, fallback: `gpt-4o-mini`)

In [7]:
# Direct extraction, OpenAI remote model
if not openai_api_key:
    print('OPENAI_API_KEY is missing in .env; skipping remote test.')
    direct_remote_openai_claims = []
else:
    direct_remote_openai_claims = run_case(
        label=f'Direct | remote | {REMOTE_MODEL}',
        model_name=REMOTE_MODEL,
        backend='remote',
        temperature=TEMPERATURE,
        verbose=VERBOSE,
    )


Direct | remote | gpt-4o-mini
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 119 claims.
Claim count: 119
01. The United States has erupted this week about what it means to live one's religion.
02. The new Religious Freedom Restoration Act in Indiana faces criticism from critics.
03. Critics say the Indiana law uses faith as a pretext to discriminate against gay people.
04. State laws regarding religious freedom have been growing since the U.S. Religious Freedom Restoration Act became law in 1993.
05. The U.S. Religious Freedom Restoration Act was designed to prohibit the federal government from "substantially burdening" a person's exercise of religion.
06. Twenty states have some version of the religious liberty law.
07. Legal controversies regarding religious liberty laws have grown.
08. Claims under state Religious Freedom Restoration Acts are "exceedingly rare."
09. Victories under state Religious Freedom Restoration Acts mostly invol

In [8]:
print("Story used for extraction:")
print()
print(textwrap.fill(story, width=100))

Story used for extraction:

The United States has seemingly erupted this week about what it means to live your religion,
especially in Indiana, where its new Religious Freedom Restoration Act faces a firestorm from
critics who say it uses faith as a pretext to discriminate against gay people. Such state laws have
been growing ever since the U.S. Religious Freedom Restoration Act became law in 1993, designed to
prohibit the federal government from "substantially burdening" a person's exercise of religion. So
far, 20 states have some version of the religious liberty law, and the legal controversies have
grown, too. Nonetheless, claims under those state RFRAs are "exceedingly rare," and victories
involved mostly religious minorities, not Christian denominations, experts say. "There is reason to
doubt whether these state-level religious liberty provisions truly provide meaningful protections
for religious believers," wrote Wayne State University law professor Christopher Lund in a 2010
ana